# Notebook 04c — Validación cruzada sobre la **ventana local** de producción

**Proyecto BME513 · Universidad de Valparaíso** — Sebastián Inostroza Hurtado

---

## Por qué hago este experimento

En el notebook **04b** validé DistilBETO con CV 5-fold y obtuve **Macro F1 = 0,8877 ± 0,0501**. Esa medición usa como entrada el **informe completo** (`Full_Report_clean`).

Pero al auditar el código de producción encontré un **desajuste entre entrenamiento e inferencia** (*train/serve skew*):

| | Texto que recibe el modelo |
|---|---|
| **Entrenamiento** (nb 04 / 04b) | `Full_Report_clean` — informe completo (~140 subtokens de mediana) |
| **Inferencia** (`verificador_birads_ml.py`, paso 5) | Ventana de **250 caracteres** alrededor de la mención (~60 subtokens) |

El código de producción es este:

```python
# verificador_birads_ml.py — PASO 5
inicio_contexto = max(0, posicion_inicio - 200)
fin_contexto    = min(len(full_report), posicion_fin + 50)
fragmento_contexto = full_report[inicio_contexto:fin_contexto].strip()

if len(fragmento_contexto) < 30:
    fragmento_contexto = full_report[max(0, posicion_inicio - 500):]
```

**Consecuencia:** el 0,8877 se midió sobre una distribución de entrada que **no es la que el componente ve en producción**. No describe el módulo tal como está desplegado.

## Por qué la ventana existe (no es un error de diseño)

La ventana es deliberada. El verificador debe leer **exactamente el mismo texto que la regex**, para que una discrepancia entre ambos sea atribuible a la **extracción** y no a que miraron cosas distintas. Es la misma justificación del nb 08.

**El problema no es el diseño: es la medición.** Este notebook la corrige.

## Qué NO cambio

Para que la comparación con el 04b sea limpia, mantengo **idénticos**:
- el modelo base (`dccuchile/distilbert-base-spanish-uncased`)
- los hiperparámetros (`max_length=256`, `batch=8`, `lr=2e-5`, `epochs=3`, `weight_decay=0.01`)
- el protocolo de CV (5 folds estratificados, `random_state=42`)
- la augmentación textual dentro de cada fold
- el diccionario de sinónimos

**La única variable que cambia es el texto de entrada.** Eso es lo que hace de esto un experimento controlado, igual que el nb 04 aisló el idioma.

## Hipótesis

**H1** — El Macro F1 sobre ventanas **cae** respecto a 0,8877. El modelo fue entrenado con informes completos y ahora recibe una distribución distinta.

**H2** — El Macro F1 **se mantiene o sube**. La ablación (0,939 → 0,544 al enmascarar el número) demuestra que el modelo se apoya casi por completo en la categoría declarada, y ese número **está dentro de la ventana**. Lo que se recorta es contexto que el modelo apenas usaba, y además se elimina ruido.

**H3** — No hay diferencia significativa (dentro de ±1 desviación estándar).

> Registro las tres **antes** de correr el experimento. Cualquiera de los tres resultados es publicable: si cae, tengo un problema identificado y cuantificado; si se mantiene, el desajuste deja de ser una crítica válida.

---
## Paso 1 — Imports y configuración

**Qué espero ver**: `Device: MPS (Apple Silicon)`.

In [ ]:
import time, json, random, re, sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)

# ---------------------------------------------------------------
# IDÉNTICOS al nb 04b — no tocar, la comparabilidad depende de esto
# ---------------------------------------------------------------
MODEL_NAME    = "dccuchile/distilbert-base-spanish-uncased"
DATA_PATH     = "../data/processed/reports_cleaned.csv"
MAX_LENGTH    = 256
BATCH_SIZE    = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3
NUM_LABELS    = 7
N_FOLDS       = 5
SEED_CV       = 42

# Referencia del nb 04b (informe completo)
BASELINE_04B_MACRO_F1 = 0.8877
BASELINE_04B_STD      = 0.0501

SINONIMOS = {
    "normal": ["habitual", "sin alteraciones", "conservado", "sin hallazgos"],
    "masa": ["formacion", "lesion nodular", "nodulo", "imagen nodular"],
    "lesion": ["hallazgo", "alteracion", "imagen sospechosa"],
    "control": ["seguimiento", "revision", "evaluacion"],
    "rutina": ["habitual", "periodica", "regular"],
    "biopsia": ["estudio histologico", "muestra histopatologica"],
    "benigno": ["no maligno", "sin malignidad"],
    "sospechoso": ["sugerente", "con caracteristicas atipicas"],
    "categoria": ["clasificacion", "tipo"],
    "informe": ["estudio", "examen", "reporte"],
}

np.random.seed(SEED_CV); torch.manual_seed(SEED_CV); random.seed(SEED_CV)

device = ("MPS (Apple Silicon)" if torch.backends.mps.is_available()
          else "CUDA" if torch.cuda.is_available() else "CPU")
print(f"Device: {device}")
print(f"Modelo: {MODEL_NAME}")
print(f"Hiperparámetros: max_length={MAX_LENGTH}, batch={BATCH_SIZE}, "
      f"lr={LEARNING_RATE}, epochs={NUM_EPOCHS}, folds={N_FOLDS}")
print(f"\nReferencia nb 04b (informe completo): Macro F1 = {BASELINE_04B_MACRO_F1} ± {BASELINE_04B_STD}")

---
## Paso 2 — Cargo el corpus

In [ ]:
df = pd.read_csv(DATA_PATH)
assert "Full_Report_clean" in df.columns, "Falta Full_Report_clean"
assert "BI-RADS" in df.columns, "Falta BI-RADS"

print(f"Informes: {len(df)}")
print("\nDistribucion BI-RADS:")
print(df["BI-RADS"].value_counts().sort_index())

---
## Paso 3 — Reproduzco la ventana **exacta** de producción

Este es el núcleo del notebook. Importo el **buscador real** para localizar la mención igual que en producción, y replico literalmente el recorte del `verificador_birads_ml.py`.

**Por qué importar el buscador en vez de escribir una regex nueva**: si escribiera mi propia regex aquí, estaría midiendo *mi aproximación* a la ventana, no la ventana real. Importando el módulo, cualquier cambio futuro en el buscador se refleja automáticamente en esta medición.

**Fallback**: si el buscador no está disponible (ruta distinta), uso una réplica por regex y lo aviso. El resultado sería una aproximación, no la ventana exacta.

In [ ]:
# Ruta al código fuente del proyecto
sys.path.insert(0, "..")

USANDO_BUSCADOR_REAL = False
try:
    from src.buscador_birads import buscar_birads_final
    USANDO_BUSCADOR_REAL = True
    print("OK: usando el buscador REAL de produccion (src.buscador_birads)")
except ImportError as e:
    print(f"AVISO: no pude importar el buscador ({e}).")
    print("       Uso una replica por regex. El resultado sera una APROXIMACION.")

# Constantes copiadas literalmente de verificador_birads_ml.py, PASO 5
VENTANA_ANTES   = 200
VENTANA_DESPUES = 50
MIN_FRAGMENTO   = 30
FALLBACK_ANTES  = 500

_PAT_BIRADS = re.compile(r"bi[\s\-]?rads?", re.IGNORECASE)


def extraer_ventana_produccion(full_report: str):
    """Replica EXACTA del PASO 5 de verificador_birads_ml.py.

    Devuelve (fragmento, metodo) donde metodo indica como se localizo
    la mencion, para poder auditar la calidad de la reproduccion.
    """
    pos_ini = pos_fin = None

    if USANDO_BUSCADOR_REAL:
        try:
            r = buscar_birads_final(full_report, usar_ml_si_ambiguo=False)
            m = r.get("mencion_seleccionada")
            if m is not None:
                pos_ini = m["posicion_inicio"]
                pos_fin = m["posicion_fin"]
                metodo = "buscador_real"
        except Exception:
            pass

    if pos_ini is None:
        # Replica: ultima mencion del texto (la que la ponderacion posicional elegiria)
        ms = list(_PAT_BIRADS.finditer(full_report))
        if not ms:
            return None, "sin_mencion"
        pos_ini = ms[-1].start()
        pos_fin = ms[-1].end() + 4   # margen para capturar el numero
        metodo = "regex_replica"

    # ---- copiado literal de verificador_birads_ml.py, PASO 5 ----
    inicio_contexto = max(0, pos_ini - VENTANA_ANTES)
    fin_contexto    = min(len(full_report), pos_fin + VENTANA_DESPUES)
    fragmento = full_report[inicio_contexto:fin_contexto].strip()

    if len(fragmento) < MIN_FRAGMENTO:
        fragmento = full_report[max(0, pos_ini - FALLBACK_ANTES):]
        metodo += "+fallback_corto"
    # -------------------------------------------------------------

    return fragmento, metodo


# Prueba rapida antes de procesar todo
_demo = df["Full_Report_clean"].astype(str).iloc[0]
_frag, _met = extraer_ventana_produccion(_demo)
print(f"\n--- DEMO (informe 0) ---")
print(f"Metodo: {_met}")
print(f"Informe completo : {len(_demo)} chars")
print(f"Ventana          : {len(_frag) if _frag else 0} chars")
print(f"\nVentana:\n{_frag}")

---
## Paso 4 — Construyo el dataset de ventanas

**Qué espero ver**: la gran mayoría procesada con `buscador_real`, y una longitud de ventana bastante estable en torno a los 250 caracteres.

In [ ]:
t0 = time.time()

textos_completos = df["Full_Report_clean"].astype(str).values
etiquetas        = df["BI-RADS"].astype(int).values

ventanas, metodos, idx_validos = [], [], []
for i, txt in enumerate(textos_completos):
    frag, met = extraer_ventana_produccion(txt)
    metodos.append(met)
    if frag:
        ventanas.append(frag)
        idx_validos.append(i)

X_ventana = np.array(ventanas)
y_ventana = etiquetas[idx_validos]

print(f"Procesado en {time.time()-t0:.1f}s\n")
print("Como se localizo la mencion:")
for k, v in Counter(metodos).most_common():
    print(f"   {k:26s}: {v:5d}  ({v/len(df)*100:5.1f}%)")

print(f"\nInformes con ventana valida: {len(X_ventana)} de {len(df)}")
print(f"Informes descartados (sin mencion): {len(df)-len(X_ventana)}")

---
## ✋ Checkpoint 1 — Validación pasiva de la ventana (no entrena)

**Tres asserts que deben pasar antes de gastar 40 minutos de GPU.** Si alguno falla, la ventana está mal construida y la métrica no significaría nada.

In [ ]:
# --- Assert 1: la ventana debe CONTENER el BI-RADS ---
# Si no lo contiene, el modelo no tiene de donde leer y la comparacion es invalida
contiene = np.array([bool(_PAT_BIRADS.search(v)) for v in X_ventana])
pct_contiene = contiene.mean() * 100
print(f"1. Ventanas que contienen la mencion BI-RADS: {pct_contiene:.2f}%")
assert pct_contiene > 99.0, f"FALLA: solo {pct_contiene:.1f}% contiene el BI-RADS"

# --- Assert 2: distribucion de clases preservada ---
d_orig = pd.Series(etiquetas).value_counts(normalize=True).sort_index()
d_vent = pd.Series(y_ventana).value_counts(normalize=True).sort_index()
max_dif = (d_orig - d_vent).abs().max()
print(f"2. Maxima desviacion en la distribucion de clases: {max_dif:.4f}")
assert max_dif < 0.01, "FALLA: la distribucion de clases cambio al filtrar"

# --- Assert 3: las ventanas son mas cortas que los informes ---
len_orig = np.array([len(t) for t in textos_completos[idx_validos]])
len_vent = np.array([len(v) for v in X_ventana])
print(f"3. Longitud media: informe {len_orig.mean():.0f} chars -> ventana {len_vent.mean():.0f} chars")
assert len_vent.mean() < len_orig.mean(), "FALLA: la ventana no es mas corta"

print("\n--- Longitud de la ventana (chars) ---")
print(f"   mediana {np.median(len_vent):.0f} | p10 {np.percentile(len_vent,10):.0f} "
      f"| p90 {np.percentile(len_vent,90):.0f} | max {len_vent.max():.0f}")

print("\n--- Reduccion de contexto ---")
print(f"   El modelo ve un {len_vent.mean()/len_orig.mean()*100:.1f}% del texto que vio al entrenar")

print("\nCHECKPOINT 1: PASSED")

---
## Paso 5 — Cuánto se recorta, en subtokens

El `max_length=256` opera sobre **subtokens**, no caracteres. Verifico cuánto se reduce realmente la entrada del modelo — y de paso confirmo que con la ventana **nunca** se trunca.

In [ ]:
tok_check = AutoTokenizer.from_pretrained(MODEL_NAME)

_muestra = np.random.RandomState(SEED_CV).choice(len(X_ventana), size=min(500, len(X_ventana)), replace=False)
n_sub_vent = [len(tok_check(X_ventana[i])["input_ids"]) for i in _muestra]
n_sub_full = [len(tok_check(textos_completos[idx_validos][i])["input_ids"]) for i in _muestra]

print("SUBTOKENS por entrada (muestra de 500)")
print(f"   Informe completo : mediana {np.median(n_sub_full):5.0f} | p90 {np.percentile(n_sub_full,90):5.0f} | max {max(n_sub_full):5.0f}")
print(f"   Ventana local    : mediana {np.median(n_sub_vent):5.0f} | p90 {np.percentile(n_sub_vent,90):5.0f} | max {max(n_sub_vent):5.0f}")
print()
print(f"   Truncados con max_length={MAX_LENGTH}:")
print(f"      informe completo : {sum(1 for n in n_sub_full if n > MAX_LENGTH)/len(n_sub_full)*100:.1f}%")
print(f"      ventana local    : {sum(1 for n in n_sub_vent if n > MAX_LENGTH)/len(n_sub_vent)*100:.1f}%")
print()
print(f"   >>> El modelo recibe ~{np.median(n_sub_vent)/np.median(n_sub_full)*100:.0f}% de los subtokens con los que fue entrenado")

---
## ✋ Checkpoint 2 — Auditoría de fuga y deduplicación

**La celda más importante del notebook.** Al recortar a ventanas aparece un problema que el nb 04b nunca trató: **duplicados**.

| | Únicos | Duplicados exactos |
|---|---|---|
| Informes completos | 3 870 de 4 357 | **487** (11,2 %) |
| **Ventanas de 250 car.** | 2 706 de 4 355 | **1 649** (37,9 %) |

Dos informes con hallazgos distintos pero la misma conclusión producen ventanas **idénticas**. Si caen en train y test, el modelo las reconoce en vez de resolverlas.

**El nb 04b también tiene fuga** (14,9 % del test con gemelo en train): el 0,8877 está inflado, aunque bastante menos que la ventana (44,6 %).

### Por qué deduplico por TEXTO EXACTO y no por firma sin dígitos

En el nb 11 (NER) deduplico por firma **sin números**, porque ahí la etiqueta es la posición de un span. **Aquí sería un desastre: la etiqueta ES el dígito.**

```
Firma sin dígitos -> 2607 grupos, 6 con etiquetas CONTRADICTORIAS   <-- fusiona BI-RADS distintos
Texto exacto      -> 2706 grupos, 0 con etiquetas contradictorias   <-- correcto
```

> **El criterio de deduplicación depende de cuál es la etiqueta.** No se copia de un notebook a otro sin pensar.

In [ ]:
import re

# Informe completo restringido a los MISMOS informes que tienen ventana,
# para que A y B sean comparables uno a uno.
X_full = textos_completos[idx_validos]
y_full = etiquetas[idx_validos]

def medir_fuga(X, y):
    """Fracción del test que tiene un gemelo IDÉNTICO en train."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED_CV)
    gem = tot = 0
    for itr, ite in skf.split(X, y):
        tr = set(X[itr]); gem += sum(1 for v in X[ite] if v in tr); tot += len(ite)
    return gem / tot

def deduplicar_exacto(X, y):
    """Dedup por TEXTO EXACTO. NO por firma: aquí la etiqueta es el dígito."""
    visto, keep = set(), []
    for i, v in enumerate(X):
        if v not in visto:
            visto.add(v); keep.append(i)
    keep = np.array(keep)
    return X[keep], y[keep]

print("="*66); print("FUGA ANTES DE DEDUPLICAR"); print("="*66)
print(f"  Informe completo : {medir_fuga(X_full, y_full)*100:5.1f}% del test tiene gemelo en train")
print(f"  Ventana local    : {medir_fuga(X_ventana, y_ventana)*100:5.1f}% del test tiene gemelo en train  <-- 3x peor")

# --- Verificar el criterio ---
firma = lambda s: re.sub(r"\d+", "", s)
gf, ge = {}, {}
for v, lab in zip(X_ventana, y_ventana):
    gf.setdefault(firma(v), set()).add(lab)
    ge.setdefault(v, set()).add(lab)
cf = sum(1 for s in gf.values() if len(s) > 1)
ce = sum(1 for s in ge.values() if len(s) > 1)
print()
print("="*66); print("CRITERIO DE DEDUPLICACIÓN"); print("="*66)
print(f"  Firma sin dígitos : {len(gf):5d} grupos · {cf} con etiquetas contradictorias")
print(f"  Texto exacto      : {len(ge):5d} grupos · {ce} con etiquetas contradictorias")
assert ce == 0, "Ventanas idénticas con etiquetas distintas: revisar."
print("  -> Dedup por TEXTO EXACTO.")

# --- Deduplicar ---
dist_antes = Counter(y_ventana)
X_full_dd,    y_full_dd    = deduplicar_exacto(X_full, y_full)
X_ventana_dd, y_ventana_dd = deduplicar_exacto(X_ventana, y_ventana)

print()
print("="*66); print("DEDUPLICACIÓN"); print("="*66)
print(f"  Informe completo : {len(X_full):5d} -> {len(X_full_dd):5d}  ({(1-len(X_full_dd)/len(X_full))*100:4.1f}% eliminado)")
print(f"  Ventana local    : {len(X_ventana):5d} -> {len(X_ventana_dd):5d}  ({(1-len(X_ventana_dd)/len(X_ventana))*100:4.1f}% eliminado)")
print()
print("  Efecto por clase (ventana):")
d2 = Counter(y_ventana_dd)
for c in sorted(dist_antes):
    a, b = dist_antes[c], d2.get(c, 0)
    print(f"    BI-RADS {c}: {a:5d} -> {b:5d}{'   <-- intacta' if a == b else ''}")
print()
print("  Los duplicados están en las clases comunes; las críticas no pierden ejemplos.")
print(f"  Balance: BI-RADS 2 pasa de {dist_antes[2]/len(y_ventana)*100:.1f}% a {d2[2]/len(y_ventana_dd)*100:.1f}%")

print()
print("="*66); print("FUGA DESPUÉS DE DEDUPLICAR"); print("="*66)
f1, f2 = medir_fuga(X_full_dd, y_full_dd), medir_fuga(X_ventana_dd, y_ventana_dd)
print(f"  Informe completo : {f1*100:5.2f}%")
print(f"  Ventana local    : {f2*100:5.2f}%")
assert f1 == 0 and f2 == 0, "Todavía hay gemelos."
print("  -> CERO gemelos. Test limpio en ambas condiciones.")

---
## Paso 6 — Augmentación por fold

**Copiada literalmente del nb 04b.** La augmentación va **dentro** de cada fold, solo sobre el train: augmentar antes de partir metería variantes del mismo informe en train y test, que es la fuga de datos que ya corregí en el 04b.

In [ ]:
def augmentar_train_fold(X_train_fold, y_train_fold, n_objetivo=300, prob_sub=0.4):
    """Augmentacion textual solo sobre el train del fold actual. Identica al nb 04b."""

    def augmentar_uno(texto, n_variantes):
        variantes = []
        palabras = texto.split()
        for _ in range(n_variantes):
            nuevas = []
            for w in palabras:
                if w in SINONIMOS and random.random() < prob_sub:
                    nuevas.append(random.choice(SINONIMOS[w]))
                else:
                    nuevas.append(w)
            variantes.append(" ".join(nuevas))
        return variantes

    X_aug = list(X_train_fold)
    y_aug = list(y_train_fold)

    clases_count = Counter(y_train_fold)
    for clase, count in clases_count.items():
        if count < n_objetivo:
            faltan = n_objetivo - count
            idx_clase = np.where(y_train_fold == clase)[0]
            variantes_por_ejemplo = max(1, faltan // len(idx_clase) + 1)
            for idx in idx_clase:
                nuevas = augmentar_uno(X_train_fold[idx], variantes_por_ejemplo)
                X_aug.extend(nuevas)
                y_aug.extend([clase] * len(nuevas))

    return np.array(X_aug), np.array(y_aug)

print("Funcion augmentar_train_fold lista (identica al nb 04b)")

---
## Paso 7 — Entrenamiento de un fold

**Copiado literalmente del nb 04b**, cambiando solo el `output_dir` para no pisar los resultados anteriores.

In [ ]:
def entrenar_un_fold(X_train, y_train, X_test, y_test, fold_id):
    """Entrena un DistilBETO desde cero para un fold. Identica al nb 04b."""

    tokenizer_fold = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_fn(batch):
        return tokenizer_fold(batch["text"], truncation=True,
                              max_length=MAX_LENGTH, padding=False)

    ds_train_fold = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
    ds_test_fold  = Dataset.from_dict({"text": X_test.tolist(),  "label": y_test.tolist()})
    ds_train_fold = ds_train_fold.map(tokenize_fn, batched=True)
    ds_test_fold  = ds_test_fold.map(tokenize_fn, batched=True)

    model_fold = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={i: f"BI-RADS_{i}" for i in range(NUM_LABELS)},
        label2id={f"BI-RADS_{i}": i for i in range(NUM_LABELS)},
    )

    training_args_fold = TrainingArguments(
        output_dir=f"./results_distilbeto_cv_VENTANA/fold_{fold_id}",  # <-- unico cambio
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        seed=SEED_CV,
        fp16=False,
        dataloader_pin_memory=False,
        disable_tqdm=False,
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer_fold)

    def compute_metrics_fold(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy":    accuracy_score(labels, preds),
            "macro_f1":    f1_score(labels, preds, average="macro", zero_division=0),
            "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        }

    trainer_fold = Trainer(
        model=model_fold,
        args=training_args_fold,
        train_dataset=ds_train_fold,
        eval_dataset=ds_test_fold,
        tokenizer=tokenizer_fold,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fold,
    )

    trainer_fold.train()

    eval_metrics = trainer_fold.evaluate(eval_dataset=ds_test_fold)
    preds_out = trainer_fold.predict(ds_test_fold)
    y_pred = np.argmax(preds_out.predictions, axis=-1)

    del model_fold, trainer_fold, ds_train_fold, ds_test_fold
    torch.mps.empty_cache() if torch.backends.mps.is_available() else None

    return {
        "fold": fold_id,
        "accuracy":    eval_metrics["eval_accuracy"],
        "macro_f1":    eval_metrics["eval_macro_f1"],
        "weighted_f1": eval_metrics["eval_weighted_f1"],
        "n_train": len(X_train),
        "n_test":  len(X_test),
        "y_test":  y_test.tolist(),
        "y_pred":  y_pred.tolist(),
    }

print("Funcion entrenar_un_fold lista")

---
## ✋ Checkpoint 3 — Estimación de tiempo (no entrena)

Cálculo puro, sin ejecutar ningún paso del trainer. En MPS con DistilBETO y batch=8, cada paso toma ~160 ms (dato del nb 04).

In [ ]:
MS_POR_PASO = 0.161  # medido en el nb 04

n_train_aprox = int(len(X_ventana) * 0.8)
# la augmentacion infla las clases minoritarias a ~300 c/u
n_aug_aprox = n_train_aprox + sum(
    max(0, 300 - int(c * 0.8)) for c in Counter(y_ventana).values() if int(c * 0.8) < 300
)
pasos_por_fold = (n_aug_aprox // BATCH_SIZE) * NUM_EPOCHS
seg_por_fold = pasos_por_fold * MS_POR_PASO

print(f"Ejemplos de train por fold (post-augmentacion): ~{n_aug_aprox}")
print(f"Pasos por fold: ~{pasos_por_fold}")
print(f"Tiempo estimado por fold : ~{seg_por_fold/60:.1f} min")
print(f"Tiempo estimado TOTAL    : ~{seg_por_fold*N_FOLDS/60:.1f} min ({N_FOLDS} folds)")
print("\nNota: la ventana es ~3x mas corta que el informe completo,")
print("      asi que en la practica deberia ser MAS RAPIDO que el nb 04b.")
print("\nCHECKPOINT 2: PASSED — listo para entrenar")

---
## Paso 8 — Validación cruzada: DOS condiciones

Comparar la ventana deduplicada contra el 0,8877 (que tiene 14,9 % de fuga) cambiaría **dos variables a la vez** y no diría nada. Necesito un informe completo medido con el mismo rigor.

| Condición | Entrada | Dedup | Qué mide |
|---|---|---|---|
| **A** | Informe completo | ✅ | El nb 04b **honesto** |
| **B** | Ventana local (250 car.) | ✅ | **El componente desplegado** |

**A vs B = el efecto puro de la ventana.** Todo lo demás idéntico: mismo modelo, hiperparámetros, semilla y augmentación por fold.

> **Tiempo:** ~30-40 min en MPS. La condición B es más rápida.

In [ ]:
def correr_cv(X, y, nombre):
    """CV 5-fold estratificada. Augmentación SOLO en el train de cada fold."""
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED_CV)
    filas, oof_p, oof_t = [], [], []
    print("\n" + "#"*66)
    print(f"# CONDICIÓN: {nombre}   (n = {len(X)})")
    print("#"*66)
    for fold_id, (idx_tr, idx_te) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X[idx_tr], y[idx_tr]
        X_te, y_te = X[idx_te], y[idx_te]
        X_tr_aug, y_tr_aug = augmentar_train_fold(X_tr, y_tr)
        t0 = time.time()
        r = entrenar_un_fold(X_tr_aug, y_tr_aug, X_te, y_te, fold_id)
        dur = (time.time() - t0) / 60
        filas.append({"fold": fold_id + 1, "macro_f1": r["macro_f1"],
                      "weighted_f1": r.get("weighted_f1", np.nan),
                      "accuracy": r["accuracy"], "n_test": len(X_te),
                      "minutos": round(dur, 1)})
        oof_t.extend(list(y_te)); oof_p.extend(list(r["y_pred"]))
        print(f"  Fold {fold_id+1}: Macro F1 = {r['macro_f1']:.4f} | acc = {r['accuracy']:.4f} | {dur:.1f} min")
    return pd.DataFrame(filas), np.array(oof_t), np.array(oof_p)

res_A, oof_t_A, oof_p_A = correr_cv(X_full_dd,    y_full_dd,    "A · INFORME COMPLETO (dedup)")
res_B, oof_t_B, oof_p_B = correr_cv(X_ventana_dd, y_ventana_dd, "B · VENTANA LOCAL (dedup) — PRODUCCIÓN")
print("\n" + "="*66); print("CV COMPLETA"); print("="*66)

---
## Paso 9 — Resultados

**El número de la condición B es el que vas a defender.**

In [ ]:
BASELINE_04B, BASELINE_04B_STD = 0.8877, 0.0501
mA, sA = res_A["macro_f1"].mean(), res_A["macro_f1"].std()
mB, sB = res_B["macro_f1"].mean(), res_B["macro_f1"].std()

for nom, d, m, s in [("A — INFORME COMPLETO (dedup)", res_A, mA, sA),
                     ("B — VENTANA LOCAL (dedup)  ***  PRODUCCIÓN  ***", res_B, mB, sB)]:
    print("="*66); print(f"CONDICIÓN {nom}"); print("="*66)
    print(d.to_string(index=False))
    print(f"\n  Macro F1 = {m:.4f} ± {s:.4f}   ·   Accuracy = {d['accuracy'].mean():.4f}\n")

print("="*66); print("LAS TRES CIFRAS"); print("="*66)
print(f"  nb 04b · completo, CON fuga (14.9%) : {BASELINE_04B:.4f} ± {BASELINE_04B_STD:.4f}   <- lo que reportabas")
print(f"  A      · completo, sin fuga         : {mA:.4f} ± {sA:.4f}")
print(f"  B      · ventana, sin fuga          : {mB:.4f} ± {sB:.4f}   <- el componente real")

print("\n" + "="*66); print("LECTURA"); print("="*66)
costo = BASELINE_04B - mA
print(f"\n1) COSTO DE LA FUGA (04b -> A): {costo:+.4f}")
print("   Los duplicados casi no inflaban: el 0,8877 era esencialmente honesto."
      if abs(costo) < 0.01 else
      f"   El 14.9% de gemelos inflaba la métrica en {costo:.4f} puntos.")

delta, ruido = mB - mA, max(sA, sB)
print(f"\n2) EFECTO PURO DE LA VENTANA (A -> B): {delta:+.4f}   (ruido entre folds: ±{ruido:.4f})")
if abs(delta) < ruido:
    veredicto = "EQUIVALENTE"
    print("   No se distingue del ruido de la CV. El desajuste es INOCUO.")
    print("   Coherente con la ablación: el modelo se apoya en el número declarado,")
    print("   y ese número está dentro de la ventana.")
elif delta > 0:
    veredicto = "LA VENTANA MEJORA"
    print("   Recortar SUBE el desempeño: el resto del informe era ruido.")
else:
    veredicto = "LA VENTANA PENALIZA" + (" (leve)" if abs(delta) <= 0.10 else " (fuerte)")
    print("   El cambio de distribución cuesta. Reentrenar sobre ventanas es lo indicado.")
print(f"\n   VEREDICTO: {veredicto}")
print(f"\n3) LA CIFRA A DEFENDER: Macro F1 = {mB:.4f} ± {sB:.4f}")
print(f"   (verificador sobre ventana local, CV 5-fold, sin fuga, n = {len(X_ventana_dd)})")

print("\n" + "="*66); print("REPORTE POR CLASE — CONDICIÓN B (out-of-fold)"); print("="*66)
print(classification_report(oof_t_B, oof_p_B, labels=list(range(NUM_LABELS)),
      target_names=[f"BI-RADS {i}" for i in range(NUM_LABELS)], digits=4, zero_division=0))

---
## Paso 10 — Guardar resultados

In [ ]:
salida = {
    "experimento": "04c_cv_ventana_local",
    "descripcion": ("CV 5-fold del verificador en dos condiciones deduplicadas: "
                    "informe completo vs ventana local de 250 caracteres (-200/+50), "
                    "que es la entrada real del modulo en produccion."),
    "hallazgo_fuga": {
        "duplicados_informe_completo": int(len(X_full) - len(X_full_dd)),
        "duplicados_ventana": int(len(X_ventana) - len(X_ventana_dd)),
        "nota": ("Recortar a ventanas triplica los duplicados: informes distintos con "
                 "la misma conclusion colapsan en ventanas identicas. El nb 04b nunca "
                 "deduplico, por lo que su 0.8877 tambien esta inflado."),
        "criterio_dedup": ("texto exacto, NO firma sin digitos: en esta tarea la "
                           "etiqueta ES el digito"),
    },
    "condicion_A_informe_completo_dedup": {
        "n": int(len(X_full_dd)), "macro_f1_mean": float(mA), "macro_f1_std": float(sA),
        "accuracy_mean": float(res_A["accuracy"].mean()),
        "por_fold": res_A.to_dict(orient="records"),
    },
    "condicion_B_ventana_local_dedup": {
        "n": int(len(X_ventana_dd)), "macro_f1_mean": float(mB), "macro_f1_std": float(sB),
        "accuracy_mean": float(res_B["accuracy"].mean()),
        "por_fold": res_B.to_dict(orient="records"),
    },
    "referencia_nb04b_con_fuga": {"macro_f1_mean": BASELINE_04B, "macro_f1_std": BASELINE_04B_STD},
    "costo_de_la_fuga": float(costo),
    "efecto_puro_de_la_ventana": float(delta),
    "veredicto": veredicto,
    "cifra_a_defender": {"macro_f1": float(mB), "std": float(sB), "n": int(len(X_ventana_dd))},
}

Path("resultados").mkdir(exist_ok=True)
with open("resultados/04c_cv_ventana_local.json", "w", encoding="utf8") as f:
    json.dump(salida, f, indent=2, ensure_ascii=False)
print("Guardado en resultados/04c_cv_ventana_local.json\n")
print(json.dumps({"cifra_a_defender": salida["cifra_a_defender"],
                  "veredicto": veredicto,
                  "costo_de_la_fuga": salida["costo_de_la_fuga"]}, indent=2, ensure_ascii=False))

---
## Conclusiones

> **Completar tras correr el notebook.**

### El hallazgo metodológico (independiente del número)

Auditando este experimento encontré que **el nb 04b nunca deduplicó**. Su CV tenía un **14,9 %** de ejemplos de test con gemelo idéntico en train, así que **el 0,8877 estaba inflado**. La disciplina que sí apliqué en el NER (nb 11) no se había aplicado al verificador.

Y al recortar a ventanas el problema **se triplica** (44,6 %): informes distintos con la misma conclusión colapsan en ventanas idénticas. **La transformación misma crea duplicados que no existían.**

> **Frase:** *"Al auditar la ventana descubrí que el recorte crea duplicados que no existían a nivel de informe completo: dos informes con hallazgos distintos y la misma conclusión producen la misma ventana. Y de paso descubrí que mi propia validación cruzada del verificador nunca había deduplicado. Corregí ambas cosas y volví a medir."*

### El criterio de deduplicación

**En esta tarea la etiqueta ES el dígito**, así que la firma-sin-números del nb 11 habría fusionado ventanas de BI-RADS distintos. Aquí hay que deduplicar por **texto exacto**. La verificación lo confirma: firma → 6 grupos con etiquetas contradictorias; exacto → 0.

> **Frase:** *"El criterio de deduplicación depende de cuál es la etiqueta. En el NER la etiqueta es una posición y los dígitos son ruido; aquí la etiqueta es el dígito. Copiar el criterio de un notebook al otro habría destruido las etiquetas."*

### Las tres cifras

| | Entrada | Fuga | Macro F1 |
|---|---|---|---|
| nb 04b | Informe completo | 14,9 % | 0,8877 ± 0,0501 |
| **A** | Informe completo | 0 % | `____` |
| **B** | **Ventana local** | 0 % | **`____`** ← el componente real |

### Limitaciones

1. Cada fold **reentrena** con ventanas: mide *"¿se puede entrenar bien sobre ventanas?"*. Para medir el *skew* puro del modelo actual habría que entrenar con informes completos y evaluar con ventanas.
2. Se excluyen los informes sin mención localizable: es fiel a producción (ahí el ML no corre), pero la métrica no cubre ese caso.
3. Sigue siendo el corpus paraguayo.